# Session 2: from a line to a probability, and the wall called XOR
**A. Real-world problem.** A spam filter must say "how likely is this message spam?", not
only "yes or no". And it must learn its settings from examples, by a rule we can check.

**Plan.** (1) the perceptron rule always finishes on separable data, (2) the sigmoid turns a
score into a probability, (3) logistic regression learns by gradient descent and we check
its gradient against finite differences, (4) the learning rate can be too small or too
large, (5) no single line can solve XOR.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the perceptron rule on separable data
Labels t = +1 or -1. Write the input with a constant 1 added, x~ = (x1, x2, 1), and the
weights with the bias included, w~ = (w1, w2, b). Rule: if `t * (w~ . x~) <= 0` (wrong or on
the line), then `w~ = w~ + t * x~`. If a line exists that separates the data with margin
gamma (the smallest distance of any point to it, measured with unit-length weights) and
every x~ has length at most R, then starting from w~ = 0 the rule makes at most (R / gamma)^2
updates and then stops (the perceptron convergence theorem).

In [ ]:
def make_separable(n, gap, seed):
    """n points uniform in [-3, 3]^2, labelled by the true line w.x + b = 0, with every point at
    least `gap` away from it. Returns X, y (0/1), unit true weights, true bias."""
    rng = np.random.default_rng(seed)
    wt = np.array([1.0, 2.0]) / np.sqrt(5.0)
    bt = 0.3
    pts = []
    while len(pts) < n:
        x = rng.uniform(-3, 3, 2)
        if abs(x @ wt + bt) >= gap:
            pts.append(x)
    X = np.array(pts)
    return X, (X @ wt + bt > 0).astype(int), wt, bt


def perceptron_fit(X, y01, max_epochs=10000):
    """Perceptron rule from w~ = 0. Returns (w~, number of updates, number of passes)."""
    Xa = np.c_[X, np.ones(len(X))]
    t = 2 * y01 - 1
    w = np.zeros(3)
    updates = 0
    for epoch in range(1, max_epochs + 1):
        changed = 0
        for x, tt in zip(Xa, t):
            if tt * (w @ x) <= 0:
                w = w + tt * x
                updates += 1
                changed += 1
        if changed == 0:
            return w, updates, epoch
    return w, updates, -1


def novikoff_bound(X, y01, wt, bt):
    """(R / gamma)^2 for the true separating line."""
    Xa = np.c_[X, np.ones(len(X))]
    ws = np.r_[wt, bt]
    ws = ws / np.linalg.norm(ws)
    gamma = np.min((2 * y01 - 1) * (Xa @ ws))
    R = np.max(np.linalg.norm(Xa, axis=1))
    return float((R / gamma) ** 2)

## Step 2: the sigmoid
`sigmoid(z) = 1 / (1 + exp(-z))` squeezes any score z into (0, 1), so we can read it as a
probability. Its slope has a neat form: `s'(z) = s(z) * (1 - s(z))`. The largest slope is
0.25, at z = 0.

In [ ]:
def sigmoid(z):
    """Numerically stable sigmoid (no overflow for large |z|)."""
    z = np.asarray(z, dtype=float)
    e = np.exp(-np.abs(z))
    return np.where(z >= 0, 1.0 / (1.0 + e), e / (1.0 + e))


def sigmoid_prime(z):
    s = sigmoid(z)
    return s * (1.0 - s)

## Step 3: logistic regression
Model: `p = sigmoid(w.x + b)`, the probability that the label is 1. Loss (cross-entropy):
`L = -mean( y log p + (1-y) log(1-p) )`. The key result: the gradient is
`dL/dw = mean( (p - y) x )` and `dL/db = mean( p - y )`. "Prediction minus truth, times the
input". Gradient descent: `w = w - lr * dL/dw`, where lr is the learning rate.

In [ ]:
def make_blobs(n, sep, seed):
    """Two overlapping Gaussian clusters (unit spread) with centres `sep` apart on the x1 axis."""
    rng = np.random.default_rng(seed)
    h = n // 2
    X = np.r_[rng.normal([-sep / 2, 0.0], 1.0, (h, 2)), rng.normal([sep / 2, 0.0], 1.0, (n - h, 2))]
    return X, np.r_[np.zeros(h), np.ones(n - h)]


def logreg_loss(w, b, X, y):
    p = np.clip(sigmoid(X @ w + b), 1e-12, 1 - 1e-12)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def logreg_grad(w, b, X, y):
    """Analytic gradient (dL/dw, dL/db)."""
    r = sigmoid(X @ w + b) - y
    return X.T @ r / len(y), float(np.mean(r))


def numeric_grad(w, b, X, y, h=1e-6):
    """Central finite differences (L(w + h) - L(w - h)) / (2h) for every parameter."""
    gw = np.zeros_like(w)
    for i in range(len(w)):
        e = np.zeros_like(w)
        e[i] = h
        gw[i] = (logreg_loss(w + e, b, X, y) - logreg_loss(w - e, b, X, y)) / (2 * h)
    gb = (logreg_loss(w, b + h, X, y) - logreg_loss(w, b - h, X, y)) / (2 * h)
    return gw, gb


def train_logreg(X, y, lr, epochs):
    """Full-batch gradient descent from zero. Returns (w, b, losses) with epochs + 1 losses."""
    w, b = np.zeros(X.shape[1]), 0.0
    losses = [logreg_loss(w, b, X, y)]
    for _ in range(epochs):
        gw, gb = logreg_grad(w, b, X, y)
        w, b = w - lr * gw, b - lr * gb
        losses.append(logreg_loss(w, b, X, y))
    return w, b, losses


def accuracy(w, b, X, y):
    return float(np.mean((X @ w + b > 0) == (y == 1)))

## Step 4: the learning rate
Too small: the loss falls but painfully slowly. Too large: each step overshoots. The
cleanest demonstration is a single weight with loss `L = w^2`: the step is
`w <- w - lr * 2w = (1 - 2 lr) w`, so |w| shrinks if lr < 1 and grows if lr > 1.

In [ ]:
def gd_quadratic(lr, steps, w0=1.0):
    """Gradient descent on L = w^2. Returns the list of w values."""
    ws = [w0]
    for _ in range(steps):
        ws.append(ws[-1] - lr * 2 * ws[-1])
    return ws


# a small fixed data set (6 + 6 points, a little overlap) that the live app also uses
APP_X = np.array([[-2, 0.5], [-1.5, -1], [-1, 1.5], [-2.5, -0.5], [-0.5, -0.5], [0.8, 0.3],
                  [2, -0.5], [1.5, 1], [1, -1.5], [2.5, 0.5], [0.5, 0.8], [-0.6, 0.2]], dtype=float)
APP_Y = np.array([0.0] * 6 + [1.0] * 6)


def epochs_to_reach(X, y, lr, target, max_epochs=500):
    """Fewest epochs after which the logistic loss is <= target (None if never within max_epochs)."""
    losses = train_logreg(X, y, lr, max_epochs)[2]
    for k, v in enumerate(losses):
        if v <= target:
            return k
    return None

## Step 5: the XOR wall
XOR: inputs (0,0) and (1,1) have label 0, inputs (0,1) and (1,0) have label 1. We try an
exhaustive grid of lines `n.x = c` (720 directions, 501 offsets, both sides) and record the
best number of the 4 points a line gets right.

In [ ]:
XOR_X = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
XOR_Y = np.array([0, 1, 1, 0])
AND_Y = np.array([0, 0, 0, 1])


def best_line_score(X, y, n_angles=720, n_offsets=501, c_range=(-2.0, 3.0)):
    """Exhaustive search over lines. Returns (best number correct, how many lines get all correct,
    total lines tried, best (angle, offset))."""
    th = np.linspace(0, 2 * np.pi, n_angles, endpoint=False)
    cs = np.linspace(c_range[0], c_range[1], n_offsets)
    N = np.c_[np.cos(th), np.sin(th)]
    proj = N @ X.T                                        # (angles, points)
    pred = proj[:, None, :] > cs[None, :, None]           # (angles, offsets, points)
    correct = (pred == (y == 1)[None, None, :]).sum(axis=2)
    i, j = np.unravel_index(np.argmax(correct), correct.shape)
    return int(correct.max()), int((correct == len(y)).sum()), int(correct.size), (float(th[i]), float(cs[j]))

## Step 6: see it

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    z = np.linspace(-6, 6, 241)
    fig, ax = plt.subplots(figsize=(6, 3.8))
    ax.plot(z, sigmoid(z), label="s(z)")
    ax.plot(z, sigmoid_prime(z), label="s'(z) = s (1 - s)")
    ax.axhline(0.25, color="gray", ls=":")
    ax.set_xlabel("score z (no unit)")
    ax.set_ylabel("value (no unit)")
    ax.set_title("Sigmoid and its slope (maximum slope 0.25 at z = 0)")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "sigmoid.png", dpi=120)

    X, y = make_blobs(100, 3.0, 0)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    for lr in (0.01, 0.1, 1.0, 100.0):
        L = train_logreg(X, y, lr, 100)[2]
        ax2.plot(L, label=f"lr = {lr:g}")
        print(f"lr = {lr:g}: loss after 100 epochs = {L[-1]:.3f}")
    ax2.set_xlabel("epoch")
    ax2.set_ylabel("cross-entropy loss (no unit)")
    ax2.set_title("Learning rate: too small is slow, too large is erratic")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "learning_rates.png", dpi=120)

    best, n_perfect, total, (th, c) = best_line_score(XOR_X, XOR_Y)
    b_and = best_line_score(XOR_X, AND_Y)
    print(f"XOR: best line gets {best} of 4; lines tried {total}; perfect lines {n_perfect}")
    print(f"AND: best line gets {b_and[0]} of 4; perfect lines {b_and[1]}")
    fig3, axs = plt.subplots(1, 2, figsize=(8, 3.8))
    for a, yy, name, res in ((axs[0], XOR_Y, "XOR", (best, n_perfect, total, (th, c))), (axs[1], AND_Y, "AND", b_and)):
        a.scatter(*XOR_X[yy == 1].T, c="tab:blue", s=90, label="label 1")
        a.scatter(*XOR_X[yy == 0].T, c="tab:orange", s=90, label="label 0")
        t, cc = res[3]
        n = np.array([np.cos(t), np.sin(t)])
        p0 = n * cc
        d = np.array([-n[1], n[0]])
        a.plot(*np.c_[p0 - 3 * d, p0 + 3 * d].T, "k--", label="best line found")
        a.set_xlim(-0.5, 1.5)
        a.set_ylim(-0.5, 1.5)
        a.set_xlabel("input x1")
        a.set_ylabel("input x2")
        a.set_title(f"{name}: best line gets {res[0]} of 4")
        a.grid(alpha=0.3)
        a.legend(fontsize=7, loc="upper center")
    fig3.tight_layout()
    fig3.savefig(OUT / "xor_wall.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** In `lesson.py`, print the number of updates of `perceptron_fit` for the seeds 0 to 4 with gap 0.3 and compare each with `novikoff_bound`. Is the bound tight?
2. **Stretch.** Run `train_logreg` on the blob data with lr = 3 and lr = 30 and print the loss after 10 epochs. Which settles first, and is the faster one always safer?
3. **Explain.** Explain why the derivative s' = s (1 - s) is largest at z = 0 and what that implies for a very large positive or negative score.

## Assessment
- **A. Conceptual.** Why do we use the smooth sigmoid with cross-entropy loss for gradient descent, instead of the step 'fire if z > 0' of Session 1 with the count of wrong answers as the loss?
- **B. Numerical.** For the single example x = (2, -1), label 0, starting from w = (0, 0), b = 0 with lr = 0.4: what are the gradient and the weights after one step? Check that the loss fell.
- **C. Graph and simulation interpretation.** In the learning-rate figure, lr = 100 reaches loss 0.2 around epoch 22 and yet ends at 0.699. What does that tell you about judging a learning rate?
- **D. Design and optimisation challenge.** Design: on the 100-point blob data (sep 3, seed 0) choose a learning rate and the smallest number of epochs that gets the loss to 0.2 or below and keeps it there up to epoch 100. Give a choice and a reason, and say what goes wrong with lr = 0.01 and lr = 100.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    # --- sigmoid ---
    checks.append(("sigmoid(0) = 0.5 and the slope there is 0.25",
                   abs(float(L.sigmoid(0.0)) - 0.5) < 1e-12 and abs(float(L.sigmoid_prime(0.0)) - 0.25) < 1e-12))
    checks.append(("worked example: sigmoid(2) = 0.8808 and s'(2) = 0.8808 * 0.1192 = 0.1050",
                   abs(float(L.sigmoid(2.0)) - 0.8808) < 1e-4 and abs(float(L.sigmoid_prime(2.0)) - 0.1050) < 1e-4))
    z = np.linspace(-8, 8, 1001)
    h = 1e-6
    num = (L.sigmoid(z + h) - L.sigmoid(z - h)) / (2 * h)
    checks.append(("s' = s (1 - s) matches a numerical derivative everywhere on [-8, 8] (error < 1e-8)",
                   float(np.max(np.abs(num - L.sigmoid_prime(z)))) < 1e-8))
    checks.append(("edge: sigmoid(-800) = 0 and sigmoid(800) = 1 without overflow",
                   float(L.sigmoid(-800.0)) == 0.0 and float(L.sigmoid(800.0)) == 1.0))
    checks.append(("sigmoid(-2) = 1 - sigmoid(2) = 0.1192", abs(float(L.sigmoid(-2.0)) - 0.1192) < 1e-4))
    # --- perceptron convergence ---
    ok, within, n_runs = True, True, 0
    for seed in range(10):
        for gap in (0.3, 0.1):
            X, y, wt, bt = L.make_separable(60, gap, seed)
            w, ups, epochs = L.perceptron_fit(X, y)
            n_runs += 1
            ok &= epochs > 0 and bool(np.all((np.c_[X, np.ones(len(X))] @ w > 0) == (y == 1)))
            within &= ups <= L.novikoff_bound(X, y, wt, bt)
    checks.append((f"perceptron rule stops with a perfect line in all {n_runs} seeded runs", ok))
    checks.append(("updates never exceed the bound (R / gamma)^2 in any run", within))
    X, y, wt, bt = L.make_separable(60, 0.3, 0)
    w, ups, epochs = L.perceptron_fit(X, y)
    checks.append(("seed 0, gap 0.3: 5 updates in 2 passes, bound 203.9",
                   ups == 5 and epochs == 2 and abs(L.novikoff_bound(X, y, wt, bt) - 203.9) < 0.05))
    X, y, wt, bt = L.make_separable(60, 0.1, 9)
    w, ups, epochs = L.perceptron_fit(X, y)
    checks.append(("seed 9, gap 0.1 (a thin gap): 29 updates in 13 passes", ups == 29 and epochs == 13))
    # --- logistic regression ---
    X1 = np.array([[1.0, 2.0]])
    y1 = np.array([1.0])
    g = L.logreg_grad(np.zeros(2), 0.0, X1, y1)
    checks.append(("worked example: one point (1, 2), label 1, w = 0, b = 0 gives p = 0.5 and gradient (-0.5, -1), -0.5",
                   np.allclose(g[0], [-0.5, -1.0]) and abs(g[1] + 0.5) < 1e-12))
    l0 = L.logreg_loss(np.zeros(2), 0.0, X1, y1)
    l1 = L.logreg_loss(np.array([0.5, 1.0]), 0.5, X1, y1)
    checks.append(("worked example: the loss falls from ln 2 = 0.6931 to 0.0486 after one step with lr = 1",
                   abs(l0 - np.log(2)) < 1e-12 and abs(l1 - 0.0486) < 1e-4))
    X, y = L.make_blobs(100, 3.0, 0)
    w = np.array([0.3, -0.2])
    b = 0.1
    ga, gb = L.logreg_grad(w, b, X, y)
    na, nb = L.numeric_grad(w, b, X, y)
    A = np.r_[ga, gb]
    Nn = np.r_[na, nb]
    rel = float(np.linalg.norm(A - Nn) / np.linalg.norm(A + Nn))
    checks.append((f"gradient (p - y) x matches finite differences (relative error {rel:.1e} < 1e-6)", rel < 1e-6))
    x2 = np.array([[2.0, -1.0]])
    y2 = np.array([0.0])
    g2 = L.logreg_grad(np.zeros(2), 0.0, x2, y2)
    w2, b2 = -0.4 * g2[0], -0.4 * g2[1]
    checks.append(("hand calculation: point (2, -1), label 0, w = 0, b = 0: p = 0.5, gradient (1, -0.5) and 0.5 for the bias",
                   np.allclose(g2[0], [1.0, -0.5]) and abs(g2[1] - 0.5) < 1e-12))
    z2 = float((np.array([2.0, -1.0]) @ w2) + b2)
    checks.append(("hand calculation: one step with lr = 0.4 gives w = (-0.4, 0.2), b = -0.2, z = -1.2, p = 0.2315 and loss 0.2633 (down from 0.6931)",
                   np.allclose(w2, [-0.4, 0.2]) and abs(b2 + 0.2) < 1e-12 and abs(z2 + 1.2) < 1e-12
                   and abs(float(L.sigmoid(z2)) - 0.2315) < 1e-4 and abs(L.logreg_loss(w2, b2, x2, y2) - 0.2633) < 1e-4))
    # --- learning rates ---
    L_small = L.train_logreg(X, y, 0.01, 100)[2]
    L_mid = L.train_logreg(X, y, 1.0, 100)[2]
    L_big = L.train_logreg(X, y, 100.0, 100)[2]
    checks.append(("lr = 0.01 is too slow: loss 0.414 after 100 epochs, still above 0.2", abs(L_small[-1] - 0.414) < 5e-4 and L_small[-1] > 0.2))
    checks.append(("lr = 1 settles at loss 0.188", abs(L_mid[-1] - 0.188) < 5e-4))
    checks.append(("lr = 100 is too large: loss 0.699 after 100 epochs, worse than the start (ln 2 = 0.693)",
                   abs(L_big[-1] - 0.699) < 5e-4 and L_big[-1] > L_big[0]))
    checks.append(("design: loss <= 0.2 first reached after 15 epochs at lr = 1, 156 at lr = 0.1, never within 500 at lr = 0.01",
                   L.epochs_to_reach(X, y, 1.0, 0.2) == 15 and L.epochs_to_reach(X, y, 0.1, 0.2) == 156
                   and L.epochs_to_reach(X, y, 0.01, 0.2) is None))
    checks.append(("trap: lr = 100 touches 0.2 at epoch 22 but ends at 0.699 (look at the whole curve)",
                   L.epochs_to_reach(X, y, 100.0, 0.2) == 22 and L_big[-1] > 0.6))
    q = L.gd_quadratic(0.4, 5)
    checks.append(("L = w^2: lr = 0.4 shrinks w by 0.2 each step (1 -> 0.2 -> 0.04)", abs(q[1] - 0.2) < 1e-12 and abs(q[2] - 0.04) < 1e-12))
    q = L.gd_quadratic(1.1, 5)
    checks.append(("L = w^2: lr = 1.1 multiplies w by -1.2 each step and diverges (|w| = 2.49 after 5 steps)",
                   abs(q[5] + 2.48832) < 1e-5))
    q = L.gd_quadratic(1.0, 4)
    checks.append(("edge: lr = 1 on L = w^2 flips the sign forever and never converges", q == [1.0, -1.0, 1.0, -1.0, 1.0]))
    # --- the app data set ---
    _, _, la = L.train_logreg(L.APP_X, L.APP_Y, 1.0, 10)
    _, _, lb = L.train_logreg(L.APP_X, L.APP_Y, 0.05, 10)
    wa, ba, lc = L.train_logreg(L.APP_X, L.APP_Y, 1.0, 100)
    checks.append(("app data set: lr = 1 reaches loss 0.353 in 10 epochs; lr = 0.05 only 0.570; the floor is 0.350",
                   abs(la[-1] - 0.353) < 5e-4 and abs(lb[-1] - 0.570) < 5e-4 and abs(lc[-1] - 0.350) < 5e-4))
    checks.append(("app data set: accuracy 10 of 12 points (0.833)", abs(L.accuracy(wa, ba, L.APP_X, L.APP_Y) - 10 / 12) < 1e-9))
    # --- XOR ---
    best, perfect, total, _ = L.best_line_score(L.XOR_X, L.XOR_Y)
    checks.append((f"XOR: exhaustive search over {total} lines finds none with 4 of 4 and the best gets {best}",
                   total == 720 * 501 and perfect == 0 and best == 3))
    bA, pA, _, _ = L.best_line_score(L.XOR_X, L.AND_Y)
    checks.append(("control: the same search finds 4 of 4 for AND (6723 perfect lines)", bA == 4 and pA == 6723))
    # hand proof: z(0,0) + z(1,1) = z(0,1) + z(1,0) for every line
    rng = np.random.default_rng(0)
    okp = True
    for _ in range(1000):
        wv, bv = rng.normal(size=2), rng.normal()
        zz = L.XOR_X @ wv + bv
        okp &= abs(zz[0] + zz[3] - zz[1] - zz[2]) < 1e-12
    checks.append(("proof step: for every line, z(0,0) + z(1,1) = z(0,1) + z(1,0)", okp))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
